<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/04_ajuste_fino_modelo_preentrenado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 04 · Ajuste fino (*fine-tuning*) de un modelo preentrenado

**Objetivo:** tomar un modelo pequeño que ya «sabe» lenguaje y adaptarlo al estilo de tus textos.

Entrenar desde cero (cuaderno 03) necesita muchísimo texto. Es mucho más eficaz partir de un modelo
**preentrenado** (que ya ha leído miles de millones de palabras) y **ajustarlo** con nuestro corpus.

- **Modelo base:** por defecto `Qwen/Qwen2.5-0.5B-Instruct` (494 millones de parámetros, buen español).
  `HuggingFaceTB/SmolLM2-135M` es más rápido, pero apenas sabe español.
- **LoRA** (*Low-Rank Adaptation*): en lugar de cambiar todos los parámetros, añadimos unas pocas
  matrices pequeñas y solo entrenamos esas (≈1–5 % del total). Ahorra memoria y tiempo y el resultado
  ocupa solo unos MB.

⏱️ Con GPU T4: 300 pasos ≈ 5–8 min (Qwen2.5-0.5B). **En CPU es muy lento** (varios segundos por paso): usa la GPU de Colab.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config, set_seed
cfg = load_config(overrides={
    "finetune": {
        # "base_model": "HuggingFaceTB/SmolLM2-135M",   # más rápido, casi sin español
        # "method": "full",                     # entrenar todos los parámetros
        # "max_steps": 50,                      # menos pasos (p. ej. en CPU)
    }
})
ajustes = cfg.finetune
set_seed(cfg.seed)
print(dict(ajustes))

ENTRENAR = True   # False = usar el adaptador ya entrenado de backups/quijote_lora (plan B, solo con Qwen2.5-0.5B-Instruct)

INICIOS = [                     # frases para comparar antes/después
    "En un lugar de la Mancha",
    "Sancho Panza le dijo a su amo",
    "Yo, señor, nací en",
]

## 1. Cargar el modelo base y los textos

In [ ]:
from dhllm.finetune import load_base_model, make_chunks
from dhllm.data import load_texts
modelo, tokenizer = load_base_model(ajustes.base_model)
textos = load_texts(cfg.data.folders)
train_chunks, val_chunks = make_chunks(textos, tokenizer, block_size=ajustes.block_size)

## 2. ¿Qué escribe el modelo ANTES del ajuste?

In [ ]:
from dhllm.finetune import generations_table, evaluate
antes = generations_table(modelo, tokenizer, INICIOS, max_new_tokens=60)
for p, t in antes.items():
    print(t, "\n" + "-" * 60)
print("Validación antes:", evaluate(modelo, val_chunks, batch_size=ajustes.batch_size, max_batches=ajustes.eval_batches))

## 3. Preparar LoRA y entrenar
La **perplejidad** indica, más o menos, entre cuántas palabras «duda» el modelo. Debería bajar.

In [ ]:
from dhllm.finetune import prepare_model, finetune_on_texts, load_adapter
if ENTRENAR:
    modelo = prepare_model(modelo, ajustes.method, ajustes.lora_r, ajustes.lora_alpha)
    historial = finetune_on_texts(modelo, train_chunks, val_chunks, ajustes)
else:
    modelo = load_adapter(modelo, "backups/quijote_lora")
    historial = None

In [ ]:
import matplotlib.pyplot as plt
if historial:
    plt.plot(historial["step"], historial["train"], marker="o", label="entrenamiento")
    plt.plot(historial["step"], historial["val"], marker="o", label="validación")
    plt.xlabel("paso"); plt.ylabel("pérdida"); plt.legend(); plt.title("Ajuste fino"); plt.show()
print("Validación después:", evaluate(modelo, val_chunks, batch_size=ajustes.batch_size, max_batches=ajustes.eval_batches))

## 4. Comparar ANTES y DESPUÉS

In [ ]:
from dhllm.finetune import compare_before_after
despues = generations_table(modelo, tokenizer, INICIOS, max_new_tokens=60)
compare_before_after(antes, despues)

## 5. Guardar
Con LoRA solo se guardan los adaptadores (unos pocos MB).

In [ ]:
from dhllm.finetune import save_model
save_model(modelo, tokenizer, ajustes.save_dir)

## 🖐️ Tu turno
1. Añade tus propios `INICIOS`.
2. Cambia el modelo base a `HuggingFaceTB/SmolLM2-135M`. ¿Qué diferencia notas en el español?
3. Entrena con tus propios textos (carpeta `data/mis_textos`). ¿Reconoces «tu» estilo?
4. Compara `"method": "lora"` y `"method": "full"`: tiempo, memoria y resultado.
5. **Para pensar:** si ajustas un modelo con las cartas de una persona, ¿de quién es el texto que genera?

> 🚀 **Para ir más lejos:** [Unsloth](https://github.com/unslothai/unsloth) ofrece cuadernos de Colab
> muy optimizados para ajustar modelos más grandes (1–8 mil millones de parámetros) con QLoRA en una T4.